In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

sample_text = "인공지능 모델이 문장의 의미를 이해하기 위해 숫자로 변환합니다."

vector_data = embeddings_model.embed_query(sample_text)

print(f"생성된 벡터의 차원 수: {len(vector_data)}")
print(f"변환된 벡터 앞부분 5개: {vector_data[:5]}")


생성된 벡터의 차원 수: 1536
변환된 벡터 앞부분 5개: [-0.00907135009765625, 0.046600341796875, -0.006256103515625, 0.0103607177734375, 0.0028438568115234375]


In [3]:
import numpy as np

# 문서 데이터베이스에 보관된 3개의 문서 청크를 정의합니다.
corpus_documents = [
    "장학금 수혜 대상은 직전 학기 평점 3.5 이상이며 F학점이 없는 학생입니다.",
    "도서관 이용 시간은 평일 오전 9시부터 오후 10시까지 운영됩니다.",
    "수강신청 변경 기간에는 최대 3학점까지 추가로 정정할 수 있습니다."
]

# 사용자의 실시간 질문을 정의합니다.
user_question = "성적 장학금을 받으려면 평점이 어떻게 되나요?"

query_vector = embeddings_model.embed_query(user_question)
doc_vector = embeddings_model.embed_documents(corpus_documents)

# cosine similarity
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

similarity_scores = [cosine_similarity(query_vector, doc_vec) for doc_vec in doc_vector]

for idx, score in enumerate(similarity_scores):
    print(f"문서 {idx+1} 유사도: {score:.4f} | 내용: {corpus_documents[idx]}")

best_doc_index = int(np.argmax(similarity_scores))
selected_context=corpus_documents[best_doc_index]
print(f"\n최종 검색된 최적 문맥: {selected_context}")


문서 1 유사도: 0.4146 | 내용: 장학금 수혜 대상은 직전 학기 평점 3.5 이상이며 F학점이 없는 학생입니다.
문서 2 유사도: 0.0797 | 내용: 도서관 이용 시간은 평일 오전 9시부터 오후 10시까지 운영됩니다.
문서 3 유사도: 0.2373 | 내용: 수강신청 변경 기간에는 최대 3학점까지 추가로 정정할 수 있습니다.

최종 검색된 최적 문맥: 장학금 수혜 대상은 직전 학기 평점 3.5 이상이며 F학점이 없는 학생입니다.


In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna"
    # temerature는 지원하지 않는 모델임
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 [참고 문서]의 내용에만 근거하여 정확하게 답변하세요.\n\n[참고 문서]:\n{context}"),
    ("user", "{question}")
])

rag_chain = prompt | model | StrOutputParser()

response = rag_chain.invoke({
    "context": selected_context,
    "question": user_question
})

print("=== [생성 답변] ===")
print(response)

=== [생성 답변] ===
직전 학기 평점이 3.5 이상이어야 하며, F학점이 없어야 합니다.
